# Tutorial 06 — Kết Luận & Khuyến Nghị Bảo Trì

> **Tóm tắt toàn bộ khóa học và hướng dẫn áp dụng thực tế tại nhà máy.**

---

## 1. Tóm Tắt Pipeline

```
Tín hiệu rung (.npy, 48kHz)
  ↓ Cắt segment (2048 mẫu, overlap 50%)
  ↓ Trích đặc trưng: Thời gian + Tần số + Envelope
  ↓ Chia train/val/test THEO FILE (GroupShuffleSplit)
  ↓ Huấn luyện SVM / Random Forest (Pipeline + GridSearchCV)
  ↓ Đánh giá (accuracy, confusion matrix, ROC-AUC)
  ↓ Giải thích bằng SHAP → Kỹ sư hiểu "vì sao"
  ↓ Ra quyết định bảo trì
```

## 2. Các Bài Học Chính

### Về Tín Hiệu
- **Envelope analysis** là công cụ mạnh nhất cho chẩn đoán ổ lăn — quan trọng hơn FFT thô.
- Tần số lỗi BPFO=107Hz, BPFI=162Hz, BSF=141Hz đều ở dải **0–500 Hz**.
- Dải 500–2000 Hz chứa harmonics, 2000–6000 Hz là vùng cộng hưởng kết cấu.

### Về Mô Hình
- **GroupShuffleSplit theo file** tránh rò rỉ dữ liệu từ overlapping segments.
- **Random Forest** thường là lựa chọn đầu tiên: dễ triển khai, cho SHAP nhanh, robust.
- Accuracy trên CWRU ≠ accuracy trên máy nhà máy (domain shift).

### Về SHAP
- SHAP waterfall phải hiển thị **giá trị vật lý gốc** (không phải z-score).
- SHAP value âm **cũng quan trọng** — nó cho biết feature đang "phản bác" lớp đó.
- Liên hệ feature → cơ chế vật lý → hành động bảo trì cụ thể.

---

## ⚠️ Giới Hạn Mô Hình — Đọc Kỹ Trước Khi Áp Dụng

### Tại Sao Accuracy 99% Trên CWRU Không Có Nghĩa Là 99% Trên Máy Của Bạn?

Mô hình CWRU được huấn luyện trong điều kiện:
- Máy thử nghiệm cố định (motor 2HP, ổ SKF 6205, cảm biến DriveEnd)
- Lỗi nhân tạo bằng EDM (rất "sạch", đơn điểm)
- Không có noise công nghiệp

### Thực Tế Nhà Máy Khác Hoàn Toàn:
- Lỗi mòn tự nhiên diễn biến dần
- Tải thay đổi → BPFO/BPFI dịch tần số theo tốc độ
- Nhiễu từ máy lân cận
- Vị trí cảm biến khác → transfer function khác

### Quy Trình Áp Dụng Đúng Cách:
1. Thu thập 2–4 tuần dữ liệu từ MÁY ĐÓ khi bình thường (baseline)
2. Khi phát hiện lỗi → thu thập dữ liệu có nhãn xác nhận
3. Xây mô hình riêng hoặc fine-tune mô hình CWRU
4. Bắt đầu với ngưỡng thủ công (RMS, kurtosis) → ML là lớp xác nhận thứ 2
5. Đặt cảm biến càng gần ổ lăn càng tốt

> **Kết luận:** CWRU là bộ dữ liệu học tập tốt nhất, nhưng mô hình cần calibrate lại trên máy thực tế.

---
*Tutorial 06 hoàn tất.*